# 02 - Run Methods

Run after `01_preprocessing.ipynb`.

Two paths, by method family:

| Methods | Where | How |
| - | - | - |
| 1a, 2, 3c, 4 (classical) | this kernel, locally | deterministic, CPU-only, no training. Reads local `data/processed/` |
| 5d, 5b (deep learning) | Modal container | needs a GPU and a checkpoint from `02a_train_dl.ipynb`. Reads the Volume |

Both write the same one-row-per-(method, fold, seed, patient) CSV, which
`03_results_analysis.ipynb` concatenates.

In [ ]:
import os, pathlib, sys

root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.paths import resolve_roots
DATA_ROOT, OUTPUT_ROOT = resolve_roots()

print('Repo root  :', root)
print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

## Classical methods (1a, 2, 3c, 4) — local

Deterministic and CPU-only, so they run in this kernel against local `data/processed/`.

In [ ]:
from src.s1_data.dataset import iter_samples, load_fold_datasets
from src.s1_data.splits import load_folds
from src.s5_eval.run_experiment import run_experiment

# Swap this import for the method under test, e.g.:
# from src.s3_methods.m1_traditional.a_thresholding import segment

PROCESSED_DIRS = (
    DATA_ROOT / 'processed/duke_dme_denoised',
    DATA_ROOT / 'processed/hc_ms_denoised',
)

FOLD = 0  # which of the 5 folds (written by 01_preprocessing.ipynb) to score on
SEED = None  # training seed; None for the deterministic methods (1a, 2, 3c, 4)
folds = load_folds(DATA_ROOT / 'processed/folds.json')
_, val_ids = folds[FOLD]

# val_ids spans both datasets. load_fold_datasets routes each id to the
# directory holding it and raises if any id is in neither (stale fold) or in
# both (same patient scored twice). iter_samples concatenates them into one
# pass, carrying subject_id so scores stay attributable to a patient.
test_set = iter_samples(load_fold_datasets(PROCESSED_DIRS, val_ids))

In [ ]:
# Results are written under OUTPUT_ROOT (local output/ for these methods).
#
# The CSV holds one row per (method, fold, seed, patient): per-patient rows are
# the unit paired comparisons consume, so re-run this per fold and let
# 03_results_analysis.ipynb concatenate.
# per_sample, per_patient, summary = run_experiment(
#     method_name='intensity_thresholding',
#     segment_fn=segment,
#     dataset=test_set,
#     fold=FOLD,
#     seed=SEED,
#     output_csv=OUTPUT_ROOT / f'intensity_thresholding_fold{FOLD}.csv',
# )
# summary

## Deep learning methods (5d, 5b) — Modal

Needs a GPU and a checkpoint written by `02a_train_dl.ipynb`, so scoring runs in a Modal
container against the Volume. Folds are scored concurrently and each writes its own CSV to
`/vol/output/`; pull them down with
`modal volume get segmentation-data /output ./output`.

In [ ]:
import modal

from src.modal_app import app, score_fold

DL_METHOD = 'unet'   # 'unet' (5d) or 'cnn_graph' (5b)
DL_FOLDS = [0]       # Stage 0 uses one fold; full sweep sets list(range(5))
DL_SEED = 42         # must match the seed 02a_train_dl.ipynb trained under

# score_fold raises FileNotFoundError if this (method, fold, seed) has no
# checkpoint on the Volume yet.
#
# Uncomment once the method's model class is implemented and trained.
# with modal.enable_output(), app.run():
#     for run in score_fold.starmap([(DL_METHOD, fold, DL_SEED) for fold in DL_FOLDS]):
#         print(f"fold {run['fold']} -> {run['csv']}")
#         print(run['summary'])